In [5]:
import pandas as pd

matches = pd.read_csv(
"../data/processed/matches_clean.csv"
)

In [6]:
matches[
"goal_diff"
] = (
matches.home_score
-
matches.away_score
)

In [7]:
def gerar_target(x):

    if x > 0:
        return 2

    elif x == 0:
        return 1

    return 0

In [8]:
matches[
"target"
] = (
matches.goal_diff
.apply(
gerar_target
)
)

In [9]:
matches[
"attack_strength"
] = (
matches.home_score
.rolling(5)
.mean()
)

In [10]:
matches[
"total_goals"
] = (
matches.home_score
+
matches.away_score
)

In [11]:
matches[
"avg_goals"
] = (
matches.total_goals
.rolling(5)
.mean()
)

In [12]:
matches = matches.dropna()

In [13]:
matches.to_csv(
"../data/processed/model_input.csv",
index=False
)

In [14]:
matches[
"home_attack_avg"
] = (

matches
.groupby(
"home_team"
)[
"home_score"
]

.transform(
lambda x:
x.shift()
.rolling(10)
.mean()
)

)

In [15]:
matches[
"away_attack_avg"
] = (

matches
.groupby(
"away_team"
)[
"away_score"
]

.transform(
lambda x:
x.shift()
.rolling(10)
.mean()
)

)

In [16]:
matches[
"home_win"
] = (
matches.home_score
>
matches.away_score
).astype(
int
)

In [17]:
matches[
"home_form"
] = (

matches
.groupby(
"home_team"
)[
"home_win"
]

.transform(
lambda x:
x.shift()
.rolling(10)
.mean()
)

)

In [18]:
matches.to_csv(
"../data/processed/model_input.csv",
index=False
)

In [19]:
import numpy as np

# substituir infinitos por NaN
matches = matches.replace(
    [np.inf, -np.inf],
    np.nan
)

# remover linhas com valores vazios
matches = matches.dropna(
    subset=[
        "home_attack_avg",
        "away_attack_avg",
        "home_form",
        "attack_strength",
        "avg_goals",
        "target"
    ]
)

print(
    matches.shape
)

print(
    matches[
        [
            "home_attack_avg",
            "away_attack_avg",
            "home_form",
            "attack_strength",
            "avg_goals"
        ]
    ].isnull().sum()
)

(44865, 21)
home_attack_avg    0
away_attack_avg    0
home_form          0
attack_strength    0
avg_goals          0
dtype: int64


In [20]:
matches.to_csv(
    "../data/processed/model_input.csv",
    index=False
)